# 第 8 章 · 记忆（ADK：`output_key`）

第二条改 state 的路：**`output_key="键名"`**——框架自动把这个 agent 的**最终答复**写进 state。

不用工具、不用写代码，一行声明即可。适合"把某个 agent 的产出存起来给后面用"。

kernel 选 `Python (agentic-adk)`。


In [1]:
# 【适配】common 已由 venv 的启动引导加进 sys.path（见 tools/_book_bootstrap.py）
from common import quiet_adk_sync_warnings, require_adk
quiet_adk_sync_warnings()
require_adk()      # 【适配】内核选错时给出可操作提示

from google.adk.agents import LlmAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai.types import Content, Part

from common import adk_model

# Define an LlmAgent with an output_key.
greeting_agent = LlmAgent(
    name="Greeter",
    model=adk_model(),          # 【适配】原书 gemini-2.0-flash 已下线
    instruction="Generate a short, friendly greeting.",
    output_key="last_greeting",
)

# --- Setup Runner and Session ---
app_name, user_id, session_id = "state_app", "user1", "session1"
session_service = InMemorySessionService()
runner = Runner(
    agent=greeting_agent,
    app_name=app_name,
    session_service=session_service,
)
session = session_service.create_session_sync(     # 【适配】2.x 该方法是异步的
    app_name=app_name, user_id=user_id, session_id=session_id)

print(f"Initial state: {session.state}")

# --- Run the Agent ---
user_message = Content(parts=[Part(text="Hello")])
print("\n--- Running the agent ---")
for event in runner.run(
    user_id=user_id,
    session_id=session_id,
    new_message=user_message,
):
    if event.is_final_response():
        print("Agent responded.")

# --- Check Updated State ---
updated_session = session_service.get_session_sync(
    app_name=app_name, user_id=user_id, session_id=session_id)
print(f"\nState after agent run: {updated_session.state}")


Initial state: {}

--- Running the agent ---


Agent responded.

State after agent run: {'last_greeting': "Hello there! I'm Greeter. It's wonderful to meet you! How can I help you today?"}
